# Week 5 Assignment #4:
Student: Daniel Foulen

Class: IS 362

Date: 2/28/2026

This notebook answers:

1) What is the northernmost airport in the United States?  
2) What is the easternmost airport in the United States?  
3) On Feb 12, 2013, which New York area airport had the windiest weather?

Data source: https://github.com/hadley/nycflights13/tree/master/data-raw
- airports.csv
- weather.csv

In [125]:
import pandas as pd
import numpy as np

## Load data

I load `airports.csv` for airport coordinates and `weather.csv` for hourly weather at NYC-area airports.

In [126]:
airports_path = "airports.csv"
weather_path  = "weather.csv"

airports = pd.read_csv(airports_path)
weather  = pd.read_csv(weather_path)

airports.head()

,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


# Identify suspicious coordinate rows 
I'll generally filter within the US's bounds.

In [127]:
sus = airports[(airports["lat"] < 15) | (airports["lat"] > 75) | (airports["lon"] < -180) | (airports["lon"] > 180) | (airports["lon"] > 0)]
sus[["faa","name","lat","lon"]].head(20)

,faa,name,lat,lon
396,DVT,Deer Valley Municipal Airport,33.411700,112.457000
417,EEN,Dillant Hopkins Airport,72.270833,42.898333
942,MYF,Montgomery Field,32.475900,117.759000
1290,SYA,Eareckson As,52.712275,174.113620


makes sense. I'll keep an eye out on these.

# Inspecting Extreme Coordinates

In [128]:
airports.sort_values("lat", ascending=False).head(10)[["faa", "name", "lat", "lon"]]

,faa,name,lat,lon
417,EEN,Dillant Hopkins Airport,72.270833,42.898333
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003
110,AIN,Wainwright Airport,70.638056,-159.994722
708,K03,Wainwright As,70.613378,-159.860350
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500
982,NUI,Nuiqsut Airport,70.210000,-151.005556
1197,SCC,Deadhorse,70.194750,-148.465167
232,BTI,Barter Island Lrrs,70.133989,-143.581867
1084,PIZ,Point Lay Lrrs,69.732875,-163.005342


# Doing a bit of research, EEN seems to have its lat and lon swapped. Let's fix that.
And by research, I mean after running through this entire notebook (and a few google searches), I realized EEN in the csv has coordinates in what's around the Arctic/Russia, when in reality it's in New Hampshire. 

So, fixing that.

In [129]:
mask = airports["faa"] == "EEN"
airports.loc[mask, ["lat", "lon"]] = airports.loc[mask, ["lon", "lat"]].to_numpy()
airports.loc[mask, "lon"] = -airports.loc[mask, "lon"]

airports[airports["faa"] == "EEN"][["faa", "name", "lat", "lon"]]

,faa,name,lat,lon
417,EEN,Dillant Hopkins Airport,42.898333,-72.270833


In [130]:
weather.head()

,origin,year,month,day,hour,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour
0,EWR,2013,1,1,1,39.02,26.06,59.37,270.0,10.35702,NaN,0.0,1012.0,10.0,2013-01-01T06:00:00Z
1,EWR,2013,1,1,2,39.02,26.96,61.63,250.0,8.05546,NaN,0.0,1012.3,10.0,2013-01-01T07:00:00Z
2,EWR,2013,1,1,3,39.02,28.04,64.43,240.0,11.50780,NaN,0.0,1012.5,10.0,2013-01-01T08:00:00Z
3,EWR,2013,1,1,4,39.92,28.04,62.21,250.0,12.65858,NaN,0.0,1012.2,10.0,2013-01-01T09:00:00Z
4,EWR,2013,1,1,5,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z


## Quick look at available columns

I inspect columns so I know what fields to use for latitude/longitude and wind.

In [131]:
airports.columns

Index(['faa', 'name', 'lat', 'lon', 'alt', 'tz', 'dst', 'tzone'], dtype='str')

In [132]:
weather.columns

Index(['origin', 'year', 'month', 'day', 'hour', 'temp', 'dewp', 'humid',
       'wind_dir', 'wind_speed', 'wind_gust', 'precip', 'pressure', 'visib',
       'time_hour'],
      dtype='str')

# Question 1: Northernmost airport in the United States

Approach:
- Use airport latitude (`lat`).
- Higher latitude means farther north.
- Restrict to U.S. airports using timezone offsets present in the dataset. U.S. airports fall within time zone offsets between –10 and –4.

In [147]:
us_airports = airports[airports["tz"].between(-10, -4, inclusive="both")].copy()

us_airports[["faa", "name", "lat", "lon", "tz", "tzone"]].head()

,faa,name,lat,lon,tz,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,-5,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,-6,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,-6,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,-5,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,-5,America/New_York


## Top 5 northernmost candidates

I sort U.S. airports by latitude (descending) and show the top 5 to make the result easy to verify.

In [134]:
north_top5 = (
    us_airports
    .dropna(subset=["lat"])
    .sort_values("lat", ascending=False)
    [["faa", "name", "lat", "lon", "tzone"]]
    .head(5)
)

north_top5

,faa,name,lat,lon,tzone
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500,America/Anchorage


## Northernmost airport (final)

I take the first row of the sorted top-5 list as the northernmost airport.

In [135]:
northernmost = north_top5.iloc[0]
northernmost

faa                             BRW
name     Wiley Post Will Rogers Mem
lat                       71.285446
lon                     -156.766003
tzone             America/Anchorage
Name: 230, dtype: object

# Question 2: Easternmost airport in the United States

Approach:
- Use airport longitude (`lon`).
- In this dataset, some far-eastern Alaska airports have **positive** longitude (crossing the 180° meridian),
  so "easternmost" is the maximum longitude value among U.S. airports.
- Show the top 5 to verify.

In [136]:
east_top5 = (
    us_airports
    .dropna(subset=["lon"])
    .sort_values("lon", ascending=False)
    [["faa", "name", "lat", "lon", "tzone"]]
    .head(5)
)

east_top5

,faa,name,lat,lon,tzone
1290,SYA,Eareckson As,52.712275,174.113620,America/Anchorage
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York
259,CAR,Caribou Muni,46.871500,-68.017917,America/New_York
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797,America/New_York


## Easternmost airport (final)

I take the first row of the sorted top-5 list as the easternmost airport.

In [137]:
easternmost = east_top5.iloc[0]

easternmost

faa                    SYA
name          Eareckson As
lat              52.712275
lon              174.11362
tzone    America/Anchorage
Name: 1290, dtype: object

# Question 3: Feb 12, 2013 (windiest New York area airport)

Approach:
- Filter `weather.csv` to 2013-02-12.
- Compare wind speeds by airport (`origin`).
- Use the maximum wind speed recorded for each airport that day.
- Remove values that are clearly impossible for airport weather readings (data outliers).

In [138]:
w = weather[
    (weather["year"] == 2013) &
    (weather["month"] == 2) &
    (weather["day"] == 12)
].copy()

w[["origin", "year", "month", "day", "hour", "wind_speed", "wind_gust"]].head()

,origin,year,month,day,hour,wind_speed,wind_gust
1006,EWR,2013,2,12,0,6.90468,NaN
1007,EWR,2013,2,12,1,9.20624,NaN
1008,EWR,2013,2,12,2,20.71404,25.31716
1009,EWR,2013,2,12,3,1048.36058,NaN
1010,EWR,2013,2,12,4,12.65858,NaN


## Compare maximum wind values by airport (before filtering)

This shows the raw maximum wind speed and gust recorded at each airport on that date.

In [139]:
w.groupby("origin")[["wind_speed", "wind_gust"]].max()

,wind_speed,wind_gust
origin,,
EWR,1048.36058,31.07106
JFK,20.71404,27.61872
LGA,23.01560,31.07106


## Filter impossible wind speed outliers

I keep wind_speed values in a realistic range (0 to 250 mph, anything outside of this is going to be a data error) and keep missing values as-is.
Then I recompute the max wind per airport.

In [140]:
w_clean = w[w["wind_speed"].between(0, 250, inclusive="both") | w["wind_speed"].isna()].copy()
w_clean.groupby("origin")[["wind_speed", "wind_gust"]].max()

,wind_speed,wind_gust
origin,,
EWR,21.86482,31.07106
JFK,20.71404,27.61872
LGA,23.01560,31.07106


## Determine the windiest airport (by wind_speed)

I compute each airport’s max wind speed for the day and pick the airport with the highest value.

In [141]:
max_wind_by_airport = w_clean.groupby("origin")["wind_speed"].max().sort_values(ascending=False)
max_wind_by_airport

origin
LGA    23.01560
EWR    21.86482
JFK    20.71404
Name: wind_speed, dtype: float64

## Windiest airport code (final)

This returns the airport code (origin) with the highest max wind_speed on 2013-02-12.

In [142]:
windiest_airport_code = max_wind_by_airport.idxmax()
windiest_airport_code

'LGA'

## Row where the maximum wind occurred

This pulls the specific hour where the windiest reading occurred for that airport.

In [143]:
windiest_row = (
    w_clean[w_clean["origin"] == windiest_airport_code]
    .sort_values("wind_speed", ascending=False)
    .head(1)
)

windiest_row[["origin", "time_hour", "wind_speed", "wind_gust", "wind_dir"]]

,origin,time_hour,wind_speed,wind_gust,wind_dir
18417,LGA,2013-02-12T07:00:00Z,23.0156,31.07106,290.0


# Final Answer Review

### Northernmost U.S. Airport

The northernmost airport in the dataset (after correcting coordinate errors) is:

**Wiley Post–Will Rogers Memorial Airport (BRW)**  
Latitude: 71.285446

Verification:
- BRW is located in Utqiagvik, Alaska.
- It is widely cited as the northernmost public airport in the United States.
- Confirmed via FAA airport listings and Wikipedia.

In [144]:
northernmost

faa                             BRW
name     Wiley Post Will Rogers Mem
lat                       71.285446
lon                     -156.766003
tzone             America/Anchorage
Name: 230, dtype: object

### Easternmost U.S. Airport

The easternmost airport (by maximum longitude in U.S. territory) is:

**Eareckson Air Station (SYA)**  
Longitude: 174.113620

Assumption:
- “Easternmost” is interpreted as the maximum longitude value among U.S. airports.
- Some Alaska airports lie near the 180° meridian and therefore have large positive longitude values.

Verification:
- SYA is located on Shemya Island in the Aleutian Islands, Alaska.
- Confirmed via FAA airport data and public records.

In [145]:
easternmost

faa                    SYA
name          Eareckson As
lat              52.712275
lon              174.11362
tzone    America/Anchorage
Name: 1290, dtype: object

### Windiest Airport:

In [146]:
windiest_airport_code

'LGA'

# Final Answers

1. **Northernmost U.S. airport:** Wiley Post–Will Rogers Memorial Airport (BRW)  
2. **Easternmost U.S. airport:** Eareckson Air Station (SYA)  
3. **Windiest NY-area airport on 2013-02-12 (by max wind_speed):** LGA